<h1><center> Laboratorio di WebScraping </h1>
<h1><center> Anno Accademico 2023-2024 </h1>
<h1><center>  Docente: Laura Ricci </h1>
<h1><center>  Lezione 18 </h1>
<h1><center>  Scale Free Networks e Power Laws</h1> 
<h1><center> 12 Aprile 2024 </h1>

## Modelli di reti: Scale Free Networks

* esistono diverse reti 
    * reti sociali
    * reti tecnologiche, come Internet
    * reti ecologiche (preda-predatore, proteine,...)
* che sono caratterizzate da
    * small world 
    * high clustering
    * esiste una grande variabilità nei gradi dei nodi, non presente nelle reti generate da **WS**
* reti con queste distribuzioni dei gradi dei nodi vengono dette **scale free networks**
    * la distribuzione dei gradi dei nodi segue una legge **power law**
* i modelli visti fino ad ora (**Erdos Renyi** e **Watts Strogatz**) non sono adatti per descrivere questa distribuzione dei nodi
* nelle slide successive vedremo
    * cosa è una distribuzione **power law**
    * un nuovo modello, quello di **Barabasi Albert**, in grado di generare reti **scale free**


## Modelli di Reti sociali

* **Watts Strogatz**
  * modella la caratteristica  **small world** di molte reti sociali
  * è il modello giusto per modellare tutte le caratteristiche di una rete sociale?
* analizziamo il sottoinsieme di **Facebook** che abbiamo introdotto in una lezione precedente   
* vedremo che **WS** riesce a cogliere la caratteristica di **Small World** della rete sociale, ma  non riesce a modellare in modo appropriato la distribuzione dei gradi dei nodi
* un nuovo modello per le reti sociali
    * **Barabasi Albert Model (BA)**

## Il dataset di Facebook

* utilizziamo il dataset di **Facebook** visto in una lezione precedente e consideriamo solo le **relazioni di amicizia**
* ricordiamo che
    * il grafo delle amicizie è  ricavato da una rete di amicizie più grande
    * sono stati considerati alcuni nodi iniziali e sono state considerate le **ego networks**
    * la rete comprende la ego network di ognuno di questi nodi
        * il nodo, i suoi amici e le relazioni di amicizia tra i suoi amici
    * le ego networks sono state poi combinate
    * il grafo risultante contiene un sottoinsieme di Facebook
* a differenza del DataSet considerato nella lezione precedente
    * i dati sono memorizzati in un **file di testo** e non in un **file csv** 
        * valori separati da uno spazio bianco
        * questo tipo di file viene indicato in genere come **edgelist**
    * l'identificatore di ogni nodo è un intero (gli identificatori sono stati parsati per rendere più facile l'elaborazione)
    

## Il dataset di Facebook

<center>
<img src="Figures/EdgeList.jpg" style="width:1200px;height:1000px;"/>

## Il dataset di Facebook

In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns
import pandas as pd
pd.read_csv('DataSet/facebook_combined.txt.gz', names=['FriendShipRelation'] )


Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'seaborn'

* se leggiamo il file come fosse un **csv**, gli identificatori di una coppia di amici viene considerata un solo valore
* necessario un metodo diverso per l'acquisizione dei dati

## Il dataset di Facebook

<code>  array = np.loadtxt(filename, dtype=int) </code>
  * utilizzata principalmente per leggere e scrivere **arrays** o **matrici** da **file di testo**
  * se non indicato, il delimitatore per separare i dati è lo spazio bianco
  * unica restrizione: ogni riga deve avere lo stesso numero di elementi
  * <code> dtype </code> indica il tipo dei dati dell'array risultato
  * restituisce un array n-dimensionale di <code> NumPy</code>: nel nostro caso, un array con **due** colonne
  * ma perchè inserire i dati in un array <code> NumPy</code>?
    * **NetworkX** possiede un metodo per costruire in grafo a partire  da un qualsasi **iterable container**
     <code>  G.add_edges_from(array) </code>    

## Il dataset di Facebook

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns 

def read_graph(filename):
    G = nx.Graph()
    array = np.loadtxt(filename, dtype=int)
    print("shape of data:",array.shape)
    print("datatype of data:",array.dtype)
    G.add_edges_from(array)
    return G
fb= read_graph('DataSet/facebook_combined.txt.gz')
n = len(fb)
m = len(fb.edges())
n, m


## Il Dataset di Facebook: grado medio dei nodi

* quale è il grado medio dei nodi?
* considerare che ogni arco incide su due nodi

In [ ]:
k = int(round(2*m/n))
k


## Il Dataset di Facebook: coefficente di clustering

* utilizziamo un **approccio approssimato**,  diverso rispetto a quello presentato in una lezione precedente, per diminuire il tempo di esecuzione
    * **random sampling**
    * ripete n volte la seguente procedura ( n passato come parametro oppure 1000 per default)
        * sceglie un nodo in maniera casuale
        * sceglie due dei suoi vicini in modo casuale
        * controlla se i vicini sono connessi
    * il **coefficente di clustering approssimato** è la frazione di triangoli trovati in questo esperimento

In [ ]:
from networkx.algorithms.approximation import average_clustering
# Impostiamo il seed random in modo che la funzione random produca tutte le volte lo stesso risultato
np.random.seed(17)
C = average_clustering(fb)
C


## L'operatore di unpacking

* nella slide successiva utilizzeremo l'**operatore di unpacking** di una lista

In [ ]:
list_days=['sunday','monday','tuesday','wednesday']
print(*list_days)


## Il Dataset di Facebook: lunghezza dei cammini

* adottiamo lo stesso approccio approssimato per calcolare la lunghezza media dei cammini
* se i dati associati ai nodi non sono importanti **G.nodes** può essere abbreviato in **list(G)**
* uso di **np.rand.choice** per selezionare due elementi casuali dalla lista
    * restituisce una lista di liste di coppie di elementi

In [ ]:
def sample_path_lengths(G, trials=1000):    
    nodes = list(G)
    # Scelta random di un insieme di coppie di nodi e calcolo della lunghezza dei cammini tra le coppie
    # trials: numero delle coppie da cui scegliere
    pairs = np.random.choice(nodes, (trials, 2))
    print(pairs[0:1])
    lengths = [nx.shortest_path_length(G, *pair) 
                               for pair in pairs]
    return lengths

def estimate_path_length(G):
    return np.mean(sample_path_lengths(G))

L = estimate_path_length(fb)
L


* clustering coefficient alto, lunghezza media dei cammini bassi: è una small world

## Modellare Facebook con WS, p=0

* costruiamo un grafo di **WS** con caratteristiche analoghe al grafo di **Facebook**
* numero di nodi **n=4039**
* numero medio di vicini **k=44**


In [ ]:
n=4039
k=44
lattice_graph = nx.watts_strogatz_graph(n, k, p=0)
print(len(lattice_graph.edges()))
print("Facebook", C)
print("Watts Strogatz",average_clustering(lattice_graph))


* il clustering coefficient è un pò più alto di quello del **DataSet di Facebook**, ma dello stesso ordine di grandezza

## Modellare Facebook con WS, p=0

In [ ]:
print ("Facebook",L) 
print("LatticeGraph",estimate_path_length(lattice_graph))


* la differenza tra le lunghezze medie dei cammini è enorme
* il risultato è atteso perchè impostando **p=0**, non abbiamo aggiunto alcun elemento di casualità

## Modellare Facebook con WS, p=1

* numero di nodi **n=4039**
* numero medio di vicini **k=44**
* probabilità di riavvolgere archi **p=1**: otteniamo una rete completamente random

In [ ]:
random_graph = nx.watts_strogatz_graph(n, k, 1)
print("Facebook",C)
print("WS",average_clustering(random_graph))
print("Facebook",L)
print("WS",estimate_path_length(random_graph))


* la lunghezza dei cammini è abbastanza confrontabile
* il coefficente di clustering è troppo basso!

## Modellare Facebook con WS, p=0.05

* valori paragonabili di clustering coefficient e lunghezza media dei cammini
* in questo caso **WS** modella perfettamente la caratteristica di **small world** del dataset reale!

In [ ]:
ws = nx.watts_strogatz_graph(n, k, 0.05, seed=15)
print("Coefficente Clustering:","Facebook", C, "WS", average_clustering(ws))
print("Lunghezza media dei cammini:","Facebook",L, "WS",estimate_path_length(ws))


## Facebook e Watts Strogatz: grado dei nodi

* analizziamo grado medio e varianza dei nodi di **Facebook** e della rete **WS** e confrontiamole
* utilizziamo le funzioni <code>mean</code> e <code>std</code> di <code>numpy</code>
    * applicate, per default,  a strutture <code>array</code> di <code>numpy</code>
    * se viene passata un'altra struttura compatibile (nel nostro caso una lista), si effettua la conversione


In [ ]:
def degrees(G):
    return [G.degree(u) for u in G]
print("Facebook", np.mean(degrees(fb)), np.mean(degrees(ws)))
print("WS", np.std(degrees(fb)), np.std(degrees(ws)))


## Facebook e Watts Strogatz: grado dei nodi

* la media dei gradi dei nodi del modello si avvicina molto alla media del grado dei nodi di Facebook
* ma la **varianza è molto diversa**!
* la varianza è molto alta per **Facebook**
    * alcuni hanno tantissimi amici, altri molto pochi
* non è altrettanto alta per **WS**

## La libreria empiricaldist

* utilizzata negli esempi successivi
* fornisce la funzione **Pmf (Probability Mass Function)**
* restituisce una **Pandas Series** che rappresenta la distribuzione di probabilità dei valori in input

In [ ]:
try:
    import empiricaldist
except ImportError:
    !pip install empiricaldist
from empiricaldist import Pmf
df_1 = Pmf.from_seq([1,2,3,4,5,6])
df_1


## La libreria empiricaldist

In [ ]:
from empiricaldist import Pmf
df_2 = Pmf.from_seq([1,2,2,2,5,6])
df_2


## Facebook e Watts Strogatz: grado dei nodi

* analizziamo e visualizziamo la **distribuzione** del grado dei nodi per **Facebook** e **WS** e confrontiamoli
* notare come un modulo possa essere installato direttamente dallo script **Python**

In [ ]:
pmf_fb = Pmf.from_seq(degrees(fb))
pmf_fb.mean(), pmf_fb.std()
    

In [ ]:
pmf_ws = Pmf.from_seq(degrees(ws))
pmf_ws.mean(), pmf_ws.std()


## Facebook e Watts Strogats: grado dei nodi

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(pmf_fb,'b',label="Facebook")
plt.xlabel('Degree')
plt.ylabel('PMF')
plt.legend(loc="upper right")
plt.subplot(1,2,2)
plt.plot(pmf_ws,'r',label="WS")
plt.xlabel('Degree')
plt.legend(loc="upper right")
plt.show()


* in **WS** la distribuzione classica è **a campana**: la media è **44**
* in  Facebook ci sono molto nodi di utenti con 1 o 2 amici, ma ci sono anche nodi che hanno più di 1000 amici!
* la distribuzione del grado dei nodi in **Facebook** è di tipo **heavy-tailed** o **power law**
    * molti nodi di grado basso e pochi nodi con grado alto, e questi hanno gradi tutti diversi tra loro

## Power Law Distribution

* è una distribuzione di probabilità definita come segue

<center>
$ PMF(k) \sim k^{-\alpha} $
</center>
 
* $\alpha$ è un parametro: in genere vale $2$ oppure $3$
* nel nostro caso, $PMF(k)$ è la frazione dei nodi di grado k
* $\sim$ indica che la $PFM$ si avvicina asintoticamente a $k^{-\alpha}$, al crescere di $k$

* la probabilità di un certo valore $k$ decresce, ma non **esponenzialmente**

* altri esempi
    * distribuzione delle popolazione delle città USA
        * la maggior parte delle città ha pochi abitanti, ma ci sono poche città con un gran numero di abitanti
    * tutte le città con un gran numero di abitanti hanno un numero diverso di abitanti
    * la probabilità di avere città con **100** abitanti è grande
    * la probabilità  doi avere una città con **1000000** è bassa, ma non nulla


## Power Law Distribution su scala logaritmica

* una power law è rappresentata come una linea in **log-log scale**
    * scala logaritmica sia sull'asse delle x che sull'asse delle y

* consideriamo la seguente **power law**
    <center>
    $ y = C \times k^{-\alpha} $
    </center>

* calcoliamo il logaritmo sia della parte destra che della parte sinistra 

    <center>
    $ log(y) = {-\alpha} \times log(k) +  log(C)  $
    </center>

* ponendo 
    * $ Y= log(y) $
    * $X = log(k)$
    * $m=-\alpha$
    * $q=log(C)$
* si ottine la retta seguente il cui coefficente angolare dipende da $\alpha$
 <center>
   $Y=mX+q$
 </center>


## Power Law Distribution: esempi

<center>
<img src="Figures/PowerLaw1.jpg" style="width:1200px;height:800px;"/>

## Power Law Distribution: esempi

<center>
<img src="Figures/PowerLaw2.jpg" style="width:1200px;height:800px;"/>

## Modellare Facebook con Watts Strogatz: grado dei nodi

* disegnamo il grafico precedente in scala <code> log log </code> 

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot([20, 1000], [5e-2, 2e-4], color='gray', linestyle='dashed')
plt.scatter(pmf_fb.index, pmf_fb,color='b',s=10, label="Facebook")
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('PMF')
plt.legend(loc="upper right")
plt.subplot(1,2,2)
plt.scatter(pmf_ws.index,pmf_ws,color='r',s=10, label="WS")
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Degree')
plt.legend(loc="upper right")
plt.show()


## Modellare Facebook con Watts Strogatz: grado dei nodi

* la rappresentazione <code> log log </code> evidenzia **la coda della distribuzione**
    * ci sono molti nodi che hanno grado tra <code> 100 </code> e <code> 1000 </code>
    * tuttavia ognuno di questi nodi ha grado diverso
    * la probabilità del singolo valore dei gradi alti  è bassa
    * invece ci sono tantissimi nodi con lo stesso valore del grado, per valori bassi
* la retta rappresenta il fitting della distribuzione

## Il modello di Barabasi Albert (BA)

* nel 1999 **Barabasi and Albert** proposero un nuovo modello per descrivere la struttura di molte reti reali il cui grado può essere descritto come una **power law**

* elementi essenziali del modello
    * **Growth**: invece considerare un numero fisso di nodi, iniziare con un numero basso di nodi e poi aggiungere vertici incrementalmente
    * **Preferential Attachment**: quando viene creato un nuovo nodo, connetterlo con maggior probabilità ai nodi esistenti che hanno
        un alto grado
* il fenomeno viene anche descritto come **"the rich get richer"**
* le reti generate con questo modello vengono dette **scale free networks**


## Il modello di Barbasi Albert (BA)

* **NetworkX** fornisce una funzione per generare grafi **BA**
* parametri
  * **n**: numero di nodi che il grafo deve avere alla fine del processo incrementale
  * **k**: numero di archi generati per connettere ogni nuovo nodo aggiunto a nodi già esistenti
    * l'aggiunta di un nodo viene fatta secondo la strategia **preferential attachment**
  * nel nostro caso si aggiunge un nodo e 22 archi a ogni round
    * 22 è il numero medio di archi per nodo, nel dataset di Facebook

In [ ]:
ba = nx.barabasi_albert_graph(4039, 22)


## BA e Facebook a confronto

In [ ]:
def degrees(G):
    return [G.degree(u) for u in G]

np.mean(degrees(fb)), np.mean(degrees(ba))


In [ ]:
np.std(degrees(fb)), np.std(degrees(ba))


In [ ]:
pmf_ba = Pmf.from_seq(degrees(ba))
pmf_ba.mean(), pmf_ba.std()


* BA descrive bene la distribuzione dei gradi dei nodi di Facebook

## BA e Facebook a confronto

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.plot(pmf_fb,'b',label="Facebook")
plt.xlabel('Degree')
plt.ylabel('PMF')
plt.legend(loc="upper right")
plt.subplot(1,2,2)
plt.plot(pmf_ba,'r',label="BA")
plt.xlabel('Degree')
plt.legend(loc="upper right")
plt.show()


## BA e Facebook a confronto

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot([20, 1000], [5e-2, 2e-4], color='gray', linestyle='dashed')
plt.scatter(pmf_fb.index, pmf_fb,color='b',s=10, label="Facebook")
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('PMF')
plt.legend(loc="upper right")
plt.subplot(1,2,2)
plt.scatter(pmf_ba.index,pmf_ba,color='r',s=10, label="BA")
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Degree')
plt.legend(loc="upper right")
plt.show()


## Algoritmo per la generazione dei grafi BA

* definiamo una funzione che operi in modo simile alla funzione **NetworkX**
<code>
ba = nx.barabasi_albert_graph(4039, 22)
</code>
* aggiunge incrementalmente nodi alla rete
    * ogni nuovo nodo è connesso a m nodi già esistenti nella rete
    * la probabilità di selezionare un nodo già esistente deve essere proporzionale al grado del nodo
* mantiene una lista **repeated_nodes** in cui ogni nodo compare tante volte quanto è il suo grado
    * in questo modo, scegliendo in modo random unifome dalla lista, i nodi con grado maggiore hanno più probabilità
      di essere selezionati

<center>
<img src="Figures/BArabasi.jpg" style="width:800px;height:800px;"/>

## Alcune funzioni di utilità: la funzione zip

* la funzione <code> zip </code> crea un iteratore aggregando elementi da più oggetti iterabili (liste,...)

In [ ]:
numbers = [1, 2, 3]
letters = ['a', 'b', 'c']
zipped = zip(numbers, letters)
zipped  # Holds an iterator object
print(type(zipped))
list(zipped)


## Alcune funzioni di utilità: list replication

In [2]:
a = [1, 2, 3] 
d = a * 3 
d


[1, 2, 3, 1, 2, 3, 1, 2, 3]

In [3]:
a=[9]
b = a * 20
b


[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]

## Algoritmo per la generazione dei grafi BA

* defininiamo un grafo di **Barabasi** che alla fine conterrà **n nodi**
* ad ogni passo si aggiungono **k nodi** che vengono connessi preferenzialente a quelli esistenti, quindi si creano ** m nuovi archi**
* seed per la tandomness

In [ ]:
import random
def barabasi_albert_graph(n, n0, k, seed=None):
    if seed is not None:
        random.seed(seed)   
    G = nx.empty_graph(n0)
    # grafo iniziale con k nodi e 0 archi
    targets = list(range(n0))
    repeated_nodes = targets
    targets = random.sample(repeated_nodes, k)
    for source in range(k, n):
        G.add_edges_from(zip([source]*k, targets))
        repeated_nodes.extend(targets)
        repeated_nodes.extend([source] * k)
        #targets = _random_subset(repeated_nodes, k)
        targets = random.sample(repeated_nodes, k)
    return G


## Algoritmo per la generazione dei grafi BA

In [ ]:
def _random_subset(repeated_nodes, k):
    # seleziona un sottoinsieme random di nodi senza ripetizione
    # repeated_nodes: lista di nodi
    #  k: diemnsione dell'insieme
    # returns: insieme di nodi
    targets = set()
    while len(targets) < k:
        x = random.choice(repeated_nodes)
        targets.add(x)
    return targets


## Algoritmo per la generazione dei grafi BA

In [ ]:
import networkx as nx
import numpy as np
ba1=barabasi_albert_graph(80,10,3)
nx.draw(ba1, with_labels=True, node_size=200, node_color="skyblue", pos=nx.spring_layout(ba1))


## Scale free Networks

* le reti che hanno il grafo dei nodi descritti da una **power law** si chiamano **scale free**

* una funzione **F** viene detta **scale free** se

<center>
$f(bx) = C(b) * f(x)$
</center>   

*  C(b) è una costante che dipende solo da b
* la "forma" della distribuzione non cambia quando si moltiplicano tutti i valori del dominio per una costante, eccetto che per un valore moltiplicativo

    
<center>
$ f(bx)=(bx)^{-\alpha} = b^{-\alpha} x^{-\alpha} =  b^{-\alpha} f(x)$
</center>  
